# Задача 1

$$
\int_a^b f(x) \, dx \approx w_1 f(x_1) + w_2 f(x_2)
$$

точная для \( f(x) = 1, x, x^2, x^3 \). Это даёт систему уравнений:

$$
\begin{cases} 
w_1 + w_2 = b - a \\ 
w_1 x_1 + w_2 x_2 = \frac{b^2 - a^2}{2} \\ 
w_1 x_1^2 + w_2 x_2^2 = \frac{b^3 - a^3}{3} \\ 
w_1 x_1^3 + w_2 x_2^3 = \frac{b^4 - a^4}{4} 
\end{cases}
$$

Для интервала \([-1, 1]\) двухточечная квадратура Гаусса имеет узлы \(\pm \frac{1}{\sqrt{3}}\) и веса 1. Для произвольного \([a, b]\) узлы преобразуются линейно:

$$
x_i = \frac{b + a}{2} + \frac{b - a}{2} \xi_i, \quad w_i = \frac{b - a}{2} \omega_i,
$$

где $\xi_1 = -1/\sqrt{3}, \;\xi_2 = 1/\sqrt{3}, \;\omega_1 = \omega_2 = 1$ .

In [3]:
import math

def gauss_2(a, b):
    xi1 = -1 / math.sqrt(3)
    xi2 = 1 / math.sqrt(3)
    
    x1 = (a + b) / 2 + (b - a) / 2 * xi1
    x2 = (a + b) / 2 + (b - a) / 2 * xi2
    
    w1 = (b - a) / 2
    w2 = (b - a) / 2
    
    return x1, x2, w1, w2


from numpy.testing import assert_allclose

x1, x2, w1, w2 = gauss_2(0, 1)

def f(x, n): 
    return x**n

for n in [0, 1, 2, 3]:
    assert_allclose(w1*f(x1, n=n) + w2*f(x2, n=n),
                    1./(n+1), atol=1e-14)



# Задча 2



$$I = \int_{0}^{1} \frac{e^x}{\sqrt{x(1-x)}} \, dx.$$


Подынтегральная функция имеет особенности при $x\to 0$ (как $1/\sqrt{x}$) и при $x\to 1$ (как $1/\sqrt{1-x}$).


$$e^x = 1 + x + \frac{x^2}{2} + O(x^3).$$


$$I = \int_0^1 \frac{1 + x}{\sqrt{x(1-x)}} \, dx + \int_0^1 \frac{e^x - 1 - x}{\sqrt{x(1-x)}} \, dx.$$



$$\int_0^1 \frac{dx}{\sqrt{x(1-x)}} = \pi, \quad \int_0^1 \frac{x\,dx}{\sqrt{x(1-x)}} = \frac{\pi}{2}.$$



$$\int_0^1 \frac{1 + x}{\sqrt{x(1-x)}} \, dx = \pi + \frac{\pi}{2} = \frac{3\pi}{2}.$$

Второе слагаемое содержит  $R(x) = e^x - 1 - x \sim \frac{x^2}{2}$ при $x\to 0$. Так как существует $x^2$, $\frac{R(x)}{\sqrt{x(1-x)}}$ не имеет особенностей на $[0,1]$ и может быть вычислена численно.




Интеграл методом трапеций:

$$I_{\text{числ}} \approx h \sum_{k=1}^{n-2} f_k.$$




In [7]:
import math

def integ(npts=10):
    h = 1.0 / (npts - 1)
    total = 0.0

    for k in range(1, npts - 1):  # внутренние точки
        x = k * h
        R = math.exp(x) - 1 - x
        if x == 0 or x == 1:
            val = 0
        else:
            val = R / math.sqrt(x * (1 - x))
        total += val

    # метод трапеций: полусумма крайних + сумма внутренних
    # крайние точки дают 0, так что просто сумма внутренних
    integral_smooth = total * h

    # добавляем точно посчитанную сингулярную часть
    I = integral_smooth + 1.5 * math.pi

    return I

ns = range(2, 100, 10)
for n in ns:
    I = integ(npts=n)
    print(f"n={n:3d}, I={I:.15f}")

n=  2, I=4.712388980384690
n= 12, I=5.199825114569391
n= 22, I=5.282451041804393
n= 32, I=5.321664453510968
n= 42, I=5.345685166400866
n= 52, I=5.362321973572546
n= 62, I=5.374717930759337
n= 72, I=5.384414296324547
n= 82, I=5.392267240192067
n= 92, I=5.398795417662112


### Оценка погрешности методом Рунге

Для метода трапеций порядок точности $p=2$. Оценка погрешности:

$$I_{\text{точн}} - I_n \approx \frac{I_n - I_{2n}}{2^2 - 1} = \frac{I_n - I_{2n}}{3},$$

где $I_n$ — значение интеграла при $n$ точках.


# Задача 4


$$I_{\text{точн}}(k) = \int_{0}^{\pi} e^{-x} \sin(kx) dx = \frac{k}{1+k^2} - \frac{e^{-\pi}(k \cos k\pi + \sin k\pi)}{1+k^2}$$

Для целых чётных $k = 2m$:  
$$I = \frac{k(1 - e^{-\pi})}{1+k^2}$$

Для целых нечётных $k = 2m+1$:  
$$I = \frac{k(1 + e^{-\pi})}{1+k^2}$$


 $f(x,k) = e^{-x}\sin(kx)$

Реализуем метод Симпсона с разбиением на $n$ частей ($n$ — чётное).

$$\int_0^\pi e^{-x}\sin(kx)dx = \int_0^\pi [e^{-x} - P_m(x)]\sin(kx)dx + \int_0^\pi P_m(x)\sin(kx)dx$$

где $P_m(x)$ — интерполяционный полином степени $m$, аппроксимирующий $e^{-x}$ на $[0,\pi]$.



Будем использовать полиномы 2-й и 3-й степени.

In [8]:
import numpy as np
import matplotlib.pyplot as plt

# Точное значение интеграла
def exact_integral(k):
    return k/(1+k**2) - np.exp(-np.pi)*(k*np.cos(k*np.pi) + np.sin(k*np.pi))/(1+k**2)

# Подынтегральная функция
def f(x, k):
    return np.exp(-x) * np.sin(k*x)

# Метод Симпсона
def simpson(f, a, b, n, k):
    # n должно быть чётным
    h = (b - a) / n
    x = np.linspace(a, b, n+1)
    y = f(x, k)
    return h/3 * (y[0] + y[-1] + 4*np.sum(y[1:-1:2]) + 2*np.sum(y[2:-1:2]))

# Построение интерполяционного полинома Лагранжа по точкам (xi, yi)
def lagrange_poly(xi, yi):
    # Возвращает функцию, вычисляющую полином в точке x
    def poly(x):
        res = 0
        n = len(xi)
        for j in range(n):
            term = yi[j]
            for m in range(n):
                if m != j:
                    term *= (x - xi[m]) / (xi[j] - xi[m])
            res += term
        return res
    return poly

# Аналитический интеграл от x^m * sin(kx) от 0 до pi
def int_xm_sin(m, k):
    # int_0^pi x^m sin(kx) dx
    if m == 0:
        return (1 - np.cos(k*np.pi))/k
    elif m == 1:
        return (np.sin(k*np.pi)/k**2 - np.pi*np.cos(k*np.pi)/k)
    elif m == 2:
        return (2*np.sin(k*np.pi)/k**3 - (np.pi**2)*np.cos(k*np.pi)/k + 2*np.pi*np.sin(k*np.pi)/k**2)
    elif m == 3:
        return (6*np.sin(k*np.pi)/k**4 - (np.pi**3)*np.cos(k*np.pi)/k + 3*np.pi**2*np.sin(k*np.pi)/k**2 - 6*np.pi*np.cos(k*np.pi)/k**3)
    else:
        # общая формула через первообразную, но для m<=3 достаточно
        raise NotImplementedError

# Основной эксперимент
k_vals = [1, 10, 100, 1000]
n_simpson_direct = [10, 100, 1000, 10000]  # для прямого счёта
n_simpson_corrected = 20  # малое n для скорректированного метода

print("k\tМетод\t\tОшибка")
for k in k_vals:
    I_ex = exact_integral(k)
    
    # Прямой метод Симпсона
    for n in n_simpson_direct:
        I_dir = simpson(f, 0, np.pi, n, k)
        err_dir = abs(I_dir - I_ex)
        print(f"{k}\tпрямой n={n}\t{err_dir:.2e}")
    
    # Метод с вычитанием полинома 2-й степени
    # узлы: 0, pi/2, pi
    xi2 = np.array([0, np.pi/2, np.pi])
    yi2 = np.exp(-xi2)
    P2 = lagrange_poly(xi2, yi2)
    
    # новая функция для численного интегрирования
    def f_corr2(x, k):
        return (np.exp(-x) - P2(x)) * np.sin(k*x)
    
    # численная часть
    I_num_corr2 = simpson(f_corr2, 0, np.pi, n_simpson_corrected, k)
    
    # аналитическая часть: интеграл от P2(x)*sin(kx)
    # P2(x) = a0 + a1*x + a2*x^2
    # найдём коэффициенты a0,a1,a2 через polyfit
    coeffs2 = np.polyfit(xi2, yi2, 2)  # возвращает [a2, a1, a0] (от старшей степени)
    a2, a1, a0 = coeffs2
    I_anal_corr2 = a0*int_xm_sin(0, k) + a1*int_xm_sin(1, k) + a2*int_xm_sin(2, k)
    
    I_total_corr2 = I_num_corr2 + I_anal_corr2
    err_corr2 = abs(I_total_corr2 - I_ex)
    print(f"{k}\tвычитание P2\t{err_corr2:.2e}")
    
    # Метод с вычитанием полинома 3-й степени
    xi3 = np.array([0, np.pi/3, 2*np.pi/3, np.pi])
    yi3 = np.exp(-xi3)
    P3 = lagrange_poly(xi3, yi3)
    
    def f_corr3(x, k):
        return (np.exp(-x) - P3(x)) * np.sin(k*x)
    
    I_num_corr3 = simpson(f_corr3, 0, np.pi, n_simpson_corrected, k)
    
    coeffs3 = np.polyfit(xi3, yi3, 3)
    a3, a2, a1, a0 = coeffs3
    I_anal_corr3 = a0*int_xm_sin(0, k) + a1*int_xm_sin(1, k) + a2*int_xm_sin(2, k) + a3*int_xm_sin(3, k)
    
    I_total_corr3 = I_num_corr3 + I_anal_corr3
    err_corr3 = abs(I_total_corr3 - I_ex)
    print(f"{k}\tвычитание P3\t{err_corr3:.2e}")
    print("-"*50)

k	Метод		Ошибка
1	прямой n=10	1.16e-04
1	прямой n=100	1.13e-08
1	прямой n=1000	1.13e-12
1	прямой n=10000	2.22e-16
1	вычитание P2	5.09e-01
1	вычитание P3	2.29e-01
--------------------------------------------------
10	прямой n=10	9.47e-02
10	прямой n=100	5.08e-06
10	прямой n=1000	5.02e-10
10	прямой n=10000	5.02e-14
10	вычитание P2	2.76e-04
10	вычитание P3	1.44e-03
--------------------------------------------------
100	прямой n=10	9.57e-03
100	прямой n=100	9.57e-03
100	прямой n=1000	5.24e-07
100	прямой n=10000	5.18e-11
100	вычитание P2	9.57e-07
100	вычитание P3	1.70e-06
--------------------------------------------------
1000	прямой n=10	9.57e-04
1000	прямой n=100	9.57e-04
1000	прямой n=1000	9.57e-04
1000	прямой n=10000	5.24e-08
1000	вычитание P2	9.57e-10
1000	вычитание P3	1.70e-09
--------------------------------------------------
